# Marine Organism Classification from Microscopic Images
VGG16 transfer learning (main model) vs a custom CNN baseline, with a leak-free train / validation / test split.

**How to run (Google Colab, free GPU):** *Runtime → Change runtime type → T4 GPU*, put your dataset zip in Google Drive, then run the cells top to bottom.
Set `LIMIT_PER_CLASS = 20` and small epoch counts first as a smoke test.

In [ ]:
# --- Colab only: mount Drive and copy the RAW (un-augmented) dataset to local disk (skip if running locally) ---
from google.colab import drive
drive.mount('/content/drive')

RAW_ZIP = "/content/drive/MyDrive/raw_images.zip"   # <-- change to your path
!mkdir -p data/raw
!unzip -q "{RAW_ZIP}" -d data/raw
!ls data/raw | head -60
# Class folders must sit DIRECTLY inside data/raw. If you see one extra folder, set RAW_DIR below to it.

## 1. Setup and configuration

In [ ]:
import json, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw):
        return x

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

# ---------------- CONFIG ----------------
RAW_DIR       = "data/raw"        # ORIGINAL images, one sub-folder per class (before any augmentation)
PROCESSED_DIR = "data/processed"  # written by this notebook (grayscale + resize)
AUG_DIR       = "data/augmented_train"
OUTPUT_DIR    = "runs"
OFFLINE_AUGMENT = True
AUG_VARIANTS    = ["dark_rot90"]  # options: rot90 rot180 rot270 flip_h flip_v dark dark_rot90 (EDIT to match your pipeline)
IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42
VAL_SIZE, TEST_SIZE = 0.15, 0.15
LIMIT_PER_CLASS = 0          # e.g. 20 for a quick smoke test; 0 = use all raw images
keras.utils.set_random_seed(SEED)

## 2. Preprocess the raw images (run once)
`RAW_DIR` must hold the **original, un-augmented** images (one folder per class).
This step only applies deterministic preprocessing (grayscale, resize to 224x224) and writes a 1:1 copy to `PROCESSED_DIR`.

> **Why the order matters:** augmentation must happen *after* the train/val/test split, and only on training images.
> If augmented copies of one image land in both train and validation/test, the model is graded on images it has effectively seen, and accuracy is inflated.

In [ ]:
RAW_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def preprocess_array(img):
    # THE preprocessing used everywhere (training data and new images): grayscale image -> 224x224
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE))

def list_raw(raw_dir):
    raw_dir = Path(raw_dir)
    class_dirs = sorted(p for p in raw_dir.iterdir() if p.is_dir())
    if not class_dirs:
        raise FileNotFoundError(f"No class folders found in {raw_dir}")
    jobs = []
    for cdir in class_dirs:
        files = sorted(f for f in cdir.rglob("*") if f.suffix.lower() in RAW_EXTS)
        if LIMIT_PER_CLASS:
            files = files[:LIMIT_PER_CLASS]
        for f in files:
            name = "__".join(f.relative_to(cdir).parts)
            jobs.append((f, Path(PROCESSED_DIR) / cdir.name / (name + ".png")))   # PNG = lossless
    return jobs

def preprocess_one(job):
    src, dst = job
    if dst.exists():                      # lets you resume an interrupted run
        return True
    img = cv2.imread(str(src), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    return bool(cv2.imwrite(str(dst), preprocess_array(img)))

jobs = list_raw(RAW_DIR)
print(f"{len(jobs)} raw images found")
with ThreadPoolExecutor(max_workers=8) as ex:
    ok = list(tqdm(ex.map(preprocess_one, jobs), total=len(jobs), desc="preprocessing"))
failed = [str(j[0]) for j, o in zip(jobs, ok) if not o]
print(f"preprocessed: {sum(ok)} | unreadable/skipped: {len(failed)}", failed[:5])

kept = [(src, dst) for (src, dst), o in zip(jobs, ok) if o]
class_names = sorted({dst.parent.name for _, dst in kept})
class_idx = {c: i for i, c in enumerate(class_names)}
NUM_CLASSES = len(class_names)
df = pd.DataFrame([(str(dst), class_idx[dst.parent.name]) for _, dst in kept], columns=["path", "label"])
df["source"] = df["path"]    # id of the ORIGINAL image; augmented copies will point back to it

## 3. Split FIRST (stratified), then inspect
One stratified train / validation / test split of the **original** images. The test set is never used for training or model selection.

In [ ]:
def make_splits(df, val_size, test_size, seed):
    counts = df["label"].value_counts()
    if (counts < 3).any():
        raise ValueError(f"Classes with fewer than 3 images can't be split: {counts[counts < 3].to_dict()}")
    train_val, test = train_test_split(df, test_size=test_size, stratify=df["label"], random_state=seed)
    train, val = train_test_split(train_val, test_size=val_size / (1 - test_size),
                                  stratify=train_val["label"], random_state=seed)
    out = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])
    return out.reset_index(drop=True)

splits = make_splits(df, VAL_SIZE, TEST_SIZE, SEED)
parts = {s: splits[splits["split"] == s] for s in ("train", "val", "test")}
print(f"{NUM_CLASSES} classes | " + " | ".join(f"{s}: {len(d)}" for s, d in parts.items()))

# class distribution
counts = df["label"].map(dict(enumerate(class_names))).value_counts()
counts.plot.bar(figsize=(14, 4), title="Images per class (originals)")
plt.ylabel("images"); plt.tight_layout(); plt.show()

# sample images
sample = df.sample(min(12, len(df)), random_state=SEED)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax in axes.flat:
    ax.axis("off")
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    ax.imshow(plt.imread(r.path), cmap="gray")
    ax.set_title(class_names[r.label], fontsize=9)
plt.tight_layout(); plt.show()

### 4. Leakage check (exact duplicates in the originals)
Catches byte-identical images that appear in both train and val/test, or under conflicting labels.

In [ ]:
RUN_DUPLICATE_CHECK = True

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if RUN_DUPLICATE_CHECK:
    chk = splits.copy()
    chk["md5"] = [md5(p) for p in chk["path"]]
    dup_rows = chk[chk.duplicated("md5", keep=False)]
    train_hashes = set(chk.loc[chk["split"] == "train", "md5"])
    evals = chk[chk["split"].isin(["val", "test"])]
    leaked = evals[evals["md5"].isin(train_hashes)]
    conflicts = dup_rows.groupby("md5")["label"].nunique()
    print(f"images checked:                          {len(chk)}")
    print(f"images with an exact duplicate:          {len(dup_rows)}")
    print(f"val/test images duplicated in train:     {len(leaked)} ({len(leaked)/max(len(evals),1):.1%} of val+test)")
    print(f"duplicate groups with conflicting labels: {int((conflicts > 1).sum())}")

## 5. Offline augmentation: TRAINING images only
Creates extra copies (darkened / rotated / flipped) of the **training** images and adds them to the training set.
Validation and test images are never augmented, and no augmented copy can leak into them because the split above was made first.

The model also applies random flips/rotations on the fly. Set `OFFLINE_AUGMENT = False` to rely on that alone: it uses no disk space and does not multiply the epoch length.

In [ ]:
AUG_OPS = {
    "rot90":  lambda im: cv2.rotate(im, cv2.ROTATE_90_CLOCKWISE),
    "rot180": lambda im: cv2.rotate(im, cv2.ROTATE_180),
    "rot270": lambda im: cv2.rotate(im, cv2.ROTATE_90_COUNTERCLOCKWISE),
    "flip_h": lambda im: cv2.flip(im, 1),
    "flip_v": lambda im: cv2.flip(im, 0),
    "dark":   lambda im: cv2.addWeighted(im, 0.7, np.zeros_like(im), 0.3, 0),   # brightness x0.7
}
AUG_OPS["dark_rot90"] = lambda im: AUG_OPS["rot90"](AUG_OPS["dark"](im))      # darken, then rotate 90 deg

def augment_train(train_df):
    jobs = [(path, Path(AUG_DIR) / class_names[label] / f"{Path(path).stem}__{v}.png", v, label)
            for path, label in zip(train_df["path"], train_df["label"]) for v in AUG_VARIANTS]

    def one(job):
        src, dst, v, _ = job
        if dst.exists():
            return True
        img = cv2.imread(src, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return False
        dst.parent.mkdir(parents=True, exist_ok=True)
        return bool(cv2.imwrite(str(dst), AUG_OPS[v](img)))

    with ThreadPoolExecutor(max_workers=8) as ex:
        ok = list(tqdm(ex.map(one, jobs), total=len(jobs), desc="augmenting train"))
    rows = [(str(dst), label, src) for (src, dst, v, label), o in zip(jobs, ok) if o]
    return pd.DataFrame(rows, columns=["path", "label", "source"]).assign(split="train")

if OFFLINE_AUGMENT and AUG_VARIANTS:
    aug_df = augment_train(splits[splits["split"] == "train"])
    splits = pd.concat([splits, aug_df], ignore_index=True)
    parts = {s: splits[splits["split"] == s] for s in ("train", "val", "test")}

# ---- leakage guard: every ORIGINAL image (and all copies of it) must live in exactly one split ----
src = {s: set(d["source"]) for s, d in parts.items()}
assert not (src["train"] & src["val"]) and not (src["train"] & src["test"]) and not (src["val"] & src["test"]), \
    "LEAK: a source image appears in more than one split"
print("OK: no source image appears in more than one split")
print(f"train: {len(parts['train'])} ({len(src['train'])} originals + {len(parts['train']) - len(src['train'])} augmented copies)"
      f" | val: {len(parts['val'])} | test: {len(parts['test'])}   (val/test are never augmented)")

## 6. Input pipeline

In [ ]:
def load_image(path, img_size=IMG_SIZE):
    # float32 RGB in [0, 255]; grayscale images are replicated to 3 channels (VGG16 needs 3)
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.cast(img, tf.float32)
    return tf.image.resize(img, (img_size, img_size))   # no-op: images are already 224x224

def build_dataset(part_df, training):
    paths = part_df["path"].tolist()
    labels = part_df["label"].to_numpy(dtype="int32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (load_image(p, IMG_SIZE), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 7. Models
* **VGG16** – ImageNet-pretrained convolutional base + new head (default `gap1024`: global average pooling → Dense 1024 → 1024 → 512 → Dropout → softmax). Base starts frozen; an optional second stage fine-tunes the last conv block.
* **Custom CNN** – 3 × (Conv + MaxPool) → Dense 256 → Dropout → softmax (baseline).

Augmentation (flips, rotations) is part of the model, so it only runs during training, never on validation/test data.

In [ ]:
@keras.utils.register_keras_serializable(package="marine")
class VGG16Preprocess(layers.Layer):
    # RGB [0,255] -> BGR with ImageNet mean subtracted (what VGG16 expects)
    def call(self, x):
        x = x[..., ::-1]
        return x - tf.constant([103.939, 116.779, 123.68], dtype=x.dtype)

def augmentation():
    return keras.Sequential([layers.RandomFlip("horizontal_and_vertical"), layers.RandomRotation(0.25)], name="augmentation")

def build_cnn(num_classes, augment=True):
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augmentation()(inputs) if augment else inputs
    x = layers.Rescaling(1.0 / 255)(x)
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 3, activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    return keras.Model(inputs, layers.Dense(num_classes, activation="softmax")(x), name="custom_cnn")

def build_vgg16(num_classes, augment=True, head="gap1024", weights="imagenet"):
    # head: gap1024 | gap | fc1024 | fc4096     weights: "imagenet" or None (train from scratch)
    base = keras.applications.VGG16(include_top=False, weights=weights, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    pretrained = weights is not None
    base.trainable = not pretrained
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augmentation()(inputs) if augment else inputs
    x = VGG16Preprocess()(x)
    x = base(x, training=False) if pretrained else base(x)
    if head == "gap1024":
        x = layers.GlobalAveragePooling2D()(x)
        for u in (1024, 1024, 512):
            x = layers.Dense(u, activation="relu")(x)
    elif head == "gap":
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation="relu")(x)
    elif head in ("fc1024", "fc4096"):
        u = 1024 if head == "fc1024" else 4096
        x = layers.Flatten()(x)
        x = layers.Dense(u, activation="relu")(x)
        x = layers.Dense(u, activation="relu")(x)
    else:
        raise ValueError(f"unknown head: {head}")
    x = layers.Dropout(0.5)(x)
    return keras.Model(inputs, layers.Dense(num_classes, activation="softmax")(x), name="vgg16_transfer")

def unfreeze_vgg16_top(model, from_layer="block5_conv1"):
    base = model.get_layer("vgg16")
    base.trainable = True
    on = False
    for layer in base.layers:
        on = on or layer.name == from_layer
        layer.trainable = on

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy",
                  metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy")])

## 8. Evaluation (held-out test set)

In [ ]:
def evaluate(model, out_dir):
    out_dir = Path(out_dir)
    ds = build_dataset(parts["test"], training=False)
    probs = model.predict(ds, verbose=1)
    y_true, y_pred = parts["test"]["label"].to_numpy(), probs.argmax(axis=1)
    labels = np.arange(NUM_CLASSES)
    k = min(5, NUM_CLASSES)
    topk = np.argsort(probs, axis=1)[:, -k:]
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    off = cm.copy(); np.fill_diagonal(off, 0)
    confusions = []
    for flat in np.argsort(off, axis=None)[::-1][:10]:
        i, j = np.unravel_index(flat, off.shape)
        if off[i, j] > 0:
            confusions.append({"true": class_names[i], "predicted": class_names[j], "count": int(off[i, j])})

    metrics = {
        "n_test_images": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        f"top{k}_accuracy": float(np.mean([t in row for t, row in zip(y_true, topk)])),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0)),
        "top_confusions": confusions,
    }
    report = classification_report(y_true, y_pred, labels=labels, target_names=class_names,
                                   output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(out_dir / "classification_report.csv")
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))

    print(json.dumps({k_: v for k_, v in metrics.items() if k_ != "top_confusions"}, indent=2))
    print("Most confused pairs:", *confusions[:5], sep="\n  ")

    norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES * 0.3), max(7, NUM_CLASSES * 0.3)))
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(class_names, rotation=90, fontsize=6); ax.set_yticklabels(class_names, fontsize=6)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Normalised confusion matrix (test set)")
    fig.colorbar(im, ax=ax, fraction=0.046); fig.tight_layout()
    fig.savefig(out_dir / "confusion_matrix.png", dpi=150); plt.show()
    return metrics

## 9. Training
`train()` fits one model and then evaluates it on the test set. Results are saved to `runs/<run_name>/`.

In [ ]:
def train(model_name="vgg16", run_name=None, head="gap1024", weights="imagenet", epochs=10,
          finetune_epochs=10, lr=1e-3, finetune_lr=1e-5, patience=3, augment=True,
          use_class_weight=False):
    run_name = run_name or model_name
    out_dir = Path(OUTPUT_DIR) / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    splits.to_csv(out_dir / "splits.csv", index=False)
    (out_dir / "class_names.json").write_text(json.dumps(class_names, indent=2))

    ds = {s: build_dataset(d, training=(s == "train")) for s, d in parts.items()}
    class_weight = None
    if use_class_weight:
        w = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=parts["train"]["label"].to_numpy())
        class_weight = {i: float(x) for i, x in enumerate(w)}

    if model_name == "cnn":
        model = build_cnn(NUM_CLASSES, augment)
    else:
        model = build_vgg16(NUM_CLASSES, augment, head, None if weights in (None, "none") else "imagenet")
    compile_model(model, lr)
    model.summary()

    def callbacks(tag):
        return [keras.callbacks.ModelCheckpoint(str(out_dir / f"best_{tag}.keras"), monitor="val_loss", save_best_only=True),
                keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
                keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(1, patience // 2), min_lr=1e-7),
                keras.callbacks.CSVLogger(str(out_dir / f"history_{tag}.csv"))]

    histories = [model.fit(ds["train"], validation_data=ds["val"], epochs=epochs,
                           class_weight=class_weight, callbacks=callbacks("stage1"))]
    if model_name == "vgg16" and finetune_epochs > 0 and weights == "imagenet":
        print("Fine-tuning: unfreezing VGG16 block5")
        unfreeze_vgg16_top(model)
        compile_model(model, finetune_lr)
        histories.append(model.fit(ds["train"], validation_data=ds["val"], epochs=finetune_epochs,
                                   class_weight=class_weight, callbacks=callbacks("stage2")))

    model.save(out_dir / "model.keras")
    (out_dir / "config.json").write_text(json.dumps(
        dict(model_name=model_name, head=head, weights=weights, epochs=epochs, finetune_epochs=finetune_epochs,
             lr=lr, finetune_lr=finetune_lr, img_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED,
             augment=augment, class_weight=use_class_weight, raw_dir=RAW_DIR, offline_augment=OFFLINE_AUGMENT, aug_variants=AUG_VARIANTS), indent=2))

    merged = {}
    for h in histories:
        for key, v in h.history.items():
            merged.setdefault(key, []).extend(v)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["loss", "accuracy"]):
        ax.plot(merged[metric], label="train"); ax.plot(merged[f"val_{metric}"], label="validation")
        if len(histories) > 1:
            ax.axvline(len(histories[0].history["loss"]) - 0.5, color="gray", ls="--", label="fine-tuning starts")
        ax.set_xlabel("epoch"); ax.set_ylabel(metric); ax.legend()
    fig.tight_layout(); fig.savefig(out_dir / "training_curves.png", dpi=150); plt.show()

    metrics = evaluate(model, out_dir)
    return model, metrics

### Main model: VGG16 transfer learning

In [ ]:
# For a smoke test use e.g. epochs=1, finetune_epochs=1 together with LIMIT_PER_CLASS = 20 above
vgg_model, vgg_metrics = train("vgg16", epochs=10, finetune_epochs=10)

### Baseline: custom CNN

In [ ]:
cnn_model, cnn_metrics = train("cnn", epochs=20)

### Optional extras
```python
train("vgg16", run_name="vgg16_from_scratch", weights="none", head="fc4096", lr=1e-4, epochs=30, finetune_epochs=0)  # no pretraining
train("vgg16", run_name="vgg16_classweights", use_class_weight=True)                                              # imbalance
```

## 10. Comparison and example predictions

In [ ]:
rows = []
for m in sorted(Path(OUTPUT_DIR).glob("*/metrics.json")):
    d = json.loads(m.read_text())
    rows.append({"run": m.parent.name, **{k: round(v, 4) for k, v in d.items() if isinstance(v, (int, float))}})
pd.DataFrame(rows)

In [ ]:
demo = parts["test"].sample(8, random_state=SEED)
batch = tf.stack([load_image(p) for p in demo["path"]])
probs = vgg_model.predict(batch, verbose=0)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, r), p in zip(axes.flat, demo.iterrows(), probs):
    k = int(p.argmax())
    ax.imshow(plt.imread(r.path), cmap="gray")
    ax.set_title(f"true: {class_names[r.label]}\npred: {class_names[k]} ({p[k]:.0%})", fontsize=9,
                 color="green" if k == r.label else "red")
    ax.axis("off")
plt.tight_layout(); plt.show()

### Predict on a new image
New images get **exactly the same preprocessing as the training data** (grayscale, resize to 224x224) and nothing else:
no darkening or rotating at prediction time.

In [ ]:
def predict_file(model, path, top_k=3):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    x = np.repeat(preprocess_array(img)[None, ..., None], 3, axis=-1).astype("float32")
    p = model.predict(x, verbose=0)[0]
    for i in np.argsort(p)[::-1][:top_k]:
        print(f"{class_names[i]:<30s} {p[i]:.1%}")

# predict_file(vgg_model, "/content/my_image.jpg")

In [ ]:
# Colab storage is wiped when the session ends: copy the results to Drive
!cp -r runs /content/drive/MyDrive/marine_runs